In [2]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict
from sklearn.metrics import mean_absolute_error, accuracy_score, r2_score, roc_auc_score
import matplotlib.pyplot as plt

class HEAPredictor:
    def __init__(self):
        self.elements = ['Ti', 'Zr', 'Nb', 'Ta', 'Mo', 'Fe']
        self.r = 8.314

        self.atomic_radius = {'Ti':1.47, 'Zr':1.60, 'Nb':1.43, 'Ta':1.43, 'Mo':1.39, 'Fe':1.26}
        self.vec_dict = {'Ti':4, 'Zr':4, 'Nb':5, 'Ta':5, 'Mo':6, 'Fe':8}
        self.tm_dict = {'Ti':1941, 'Zr':2128, 'Nb':2750, 'Ta':3290, 'Mo':2896, 'Fe':1811}

        self.feature_cols = self.elements + ['delta', 'Smix', 'omega', 'VEC']
        self.aug_feature_cols = self.feature_cols + ['Laves_pred']

        self.clf_laves = None
        self.reg_modulus = None
        self.is_trained = False

    def _compute_physics(self, comp_percent):
        c = np.array(comp_percent) / 100.0

        r_avg = sum(c[i] * self.atomic_radius[e] for i, e in enumerate(self.elements))
        delta = np.sqrt(sum(c[i] * (1 - self.atomic_radius[e]/r_avg)**2 for i, e in enumerate(self.elements)))

        vec = sum(c[i] * self.vec_dict[e] for i, e in enumerate(self.elements))
        smix = -self.r * sum(ci * np.log(ci) for ci in c if ci > 0)

        tm = sum(c[i] * self.tm_dict[e] for i, e in enumerate(self.elements))
        omega = (tm * smix) / 1000

        return [delta, smix, omega, vec]

    def prepare_dataset(self, df_raw):
        features = []
        for _, row in df_raw.iterrows():
            comp = [row[e] for e in self.elements]
            phys = self._compute_physics(comp)
            features.append(comp + phys)
        return pd.DataFrame(features, columns=self.feature_cols)

    def train(self, X_raw, y_modulus, y_laves):
        X = self.prepare_dataset(X_raw)

        # Hold-out real
        X_train, X_test, y_m_train, y_m_test, y_l_train, y_l_test = train_test_split(
            X, y_modulus, y_laves, test_size=0.2, random_state=42
        )

        # CV adaptativo (evita erro com dataset pequeno)
        cv_splits = min(5, len(X_train))
        if cv_splits < 2:
            raise ValueError("Dataset pequeno demais para validação.")

        print(f">>> Usando {cv_splits}-fold CV")

        # ======================
        # CLASSIFICADOR
        # ======================
        params_clf = {'n_estimators': [100, 200], 'max_depth': [None, 10]}

        grid_clf = GridSearchCV(
            RandomForestClassifier(random_state=42, class_weight='balanced'),
            params_clf,
            cv=cv_splits
        )
        grid_clf.fit(X_train, y_l_train)
        self.clf_laves = grid_clf.best_estimator_

        # ======================
        # PREDIÇÕES SEM LEAKAGE
        # ======================
        laves_cv_preds = cross_val_predict(
            self.clf_laves,
            X_train,
            y_l_train,
            cv=cv_splits
        )

        X_train_aug = X_train.copy()
        X_train_aug['Laves_pred'] = laves_cv_preds

        # ======================
        # REGRESSOR
        # ======================
        params_reg = {'n_estimators': [100, 200], 'max_depth': [None, 10]}

        grid_reg = GridSearchCV(
            RandomForestRegressor(random_state=42),
            params_reg,
            cv=cv_splits
        )
        grid_reg.fit(X_train_aug, y_m_train)
        self.reg_modulus = grid_reg.best_estimator_

        # ======================
        # TESTE FINAL
        # ======================
        l_test_pred = self.clf_laves.predict(X_test)
        l_test_proba = self.clf_laves.predict_proba(X_test)[:, 1]

        X_test_aug = X_test.copy()
        X_test_aug['Laves_pred'] = l_test_pred

        m_test_pred = self.reg_modulus.predict(X_test_aug)

        print("\n" + "="*40)
        print("VALIDAÇÃO FINAL (Hold-out):")
        print(f"Acurácia Laves: {accuracy_score(y_l_test, l_test_pred):.2%}")

        try:
            print(f"AUC Laves: {roc_auc_score(y_l_test, l_test_proba):.4f}")
        except:
            print("AUC não calculável (classe única no teste)")

        print(f"R² Módulo: {r2_score(y_m_test, m_test_pred):.4f}")
        print(f"MAE Módulo: {mean_absolute_error(y_m_test, m_test_pred):.2f} GPa")
        print("="*40 + "\n")

        self.is_trained = True

    def predict_from_dict(self, comp_dict):
        if not self.is_trained:
            raise Exception("Modelo não treinado.")

        ordered = [comp_dict.get(el, 0) for el in self.elements]
        total = sum(ordered)

        if total == 0:
            return "Erro: composição inválida."

        norm = [100 * x / total for x in ordered]
        phys = self._compute_physics(norm)

        X = pd.DataFrame([norm + phys], columns=self.feature_cols)

        laves = self.clf_laves.predict(X)[0]

        X_aug = X.copy()
        X_aug['Laves_pred'] = laves

        modulus = self.reg_modulus.predict(X_aug)[0]

        return {
            "Laves": bool(laves),
            "Modulo (GPa)": round(modulus, 2),
            "Parametros": dict(zip(['delta', 'Smix', 'omega', 'VEC'], phys))
        }

    def save(self, path="hea_model.pkl"):
        joblib.dump(self, path)